In [0]:
# Connect Databricks to ADLS Gen2
storage_account_name = "sttelcomchurn"
storage_account_key  =  "YOUR_STORAGE_KEY_HERE"



spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("ADLS connected successfully")

ADLS connected successfully


In [0]:
# Define base path
bronze_path = "abfss://bronze@sttelcomchurn.dfs.core.windows.net/"

# Read all 3 source files
customer_df = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path + "customer_profile.csv")
billing_df  = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path + "billing_data.csv")
cdr_df      = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path + "cdr_simulation.csv")

# Verify row counts
print(f"customer_profile : {customer_df.count()} rows")
print(f"billing_data     : {billing_df.count()} rows")
print(f"cdr_simulation   : {cdr_df.count()} rows")

customer_profile : 2000000 rows
billing_data     : 2000000 rows
cdr_simulation   : 2000000 rows


In [0]:
from pyspark.sql import functions as F

# ── Silver: clean customer_df ──────────────────────────────
customer_silver = customer_df \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id", "tenure_months"]) \
    .withColumn("churned", F.col("churned").cast("int"))

# ── Silver: clean billing_df ──────────────────────────────
billing_silver = billing_df \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"]) \
    .withColumn("monthly_charges", F.col("monthly_charges").cast("float")) \
    .withColumn("total_charges",   F.col("total_charges").cast("float")) \
    .fillna({"total_charges": 0.0})

# Billing spike flag: monthly charge > 2x average
avg_charge = billing_silver.agg(F.avg("monthly_charges")).collect()[0][0]
billing_silver = billing_silver.withColumn(
    "billing_spike_flag",
    F.when(F.col("monthly_charges") > 2 * avg_charge, 1).otherwise(0)
)

# ── Silver: clean cdr_df ──────────────────────────────────
cdr_silver = cdr_df \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"]) \
    .withColumn("drop_rate", F.col("drop_rate").cast("float"))

# Verify
print(f"customer_silver  : {customer_silver.count()} rows")
print(f"billing_silver   : {billing_silver.count()} rows")
print(f"cdr_silver       : {cdr_silver.count()} rows")

customer_silver  : 2000000 rows
billing_silver   : 2000000 rows
cdr_silver       : 2000000 rows


In [0]:
# ── Gold: join all 3 silver tables ────────────────────────
gold_df = customer_silver \
    .join(billing_silver, on="customer_id", how="left") \
    .join(cdr_silver,     on="customer_id", how="left")

# ── Gold: feature engineering ──────────────────────────────
gold_df = gold_df.withColumn(
    "churn_risk_score",
    # Signal 1: month-to-month contract (your observation)
    F.when(F.col("contract_type") == "Month-to-month", 35).otherwise(0) +
    # Signal 2: tenure under 18 months (your observation)
    F.when(F.col("tenure_months") < 18, 25).otherwise(0) +
    # Signal 3: monthly charges above 74 (your observation)
    F.when(F.col("monthly_charges") > 74, 20).otherwise(0) +
    # Signal 4: billing spike flag
    (F.col("billing_spike_flag") * 20)
)

# ── Gold: churn risk label ─────────────────────────────────
gold_df = gold_df.withColumn(
    "churn_risk_label",
    F.when(F.col("churn_risk_score") >= 60, "High")
     .when(F.col("churn_risk_score") >= 30, "Medium")
     .otherwise("Low")
)

# ── Verify output ──────────────────────────────────────────
print(f"Gold table rows  : {gold_df.count()}")
print("\nChurn risk distribution:")
gold_df.groupBy("churn_risk_label") \
       .count() \
       .orderBy("churn_risk_label") \
       .show()

print("\nSample of scored customers:")
display(gold_df.select(
    "customer_id",
    "contract_type",
    "tenure_months",
    "monthly_charges",
    "billing_spike_flag",
    "churn_risk_score",
    "churn_risk_label",
    "churned"
).limit(10))

Gold table rows  : 2000000

Churn risk distribution:
+----------------+------+
|churn_risk_label| count|
+----------------+------+
|            High|834842|
|             Low|795385|
|          Medium|369773|
+----------------+------+


Sample of scored customers:


customer_id,contract_type,tenure_months,monthly_charges,billing_spike_flag,churn_risk_score,churn_risk_label,churned
CUST0000002,Two year,33,22.64,0,0,Low,0
CUST0000004,One year,56,47.02,0,0,Low,0
CUST0000011,Month-to-month,13,200.0,1,100,High,1
CUST0000012,Two year,60,68.8,0,0,Low,0
CUST0000013,Two year,30,73.31,0,0,Low,0
CUST0000014,Month-to-month,8,91.87,0,80,High,1
CUST0000020,Month-to-month,21,62.7,0,35,Medium,1
CUST0000022,Month-to-month,23,200.0,1,75,High,0
CUST0000023,Month-to-month,15,99.18,0,80,High,0
CUST0000034,Two year,8,88.43,0,45,Medium,0


In [0]:
# ── Write gold to ADLS as Parquet ─────────────────────────
gold_path = "abfss://gold@sttelcomchurn.dfs.core.windows.net/fact_churn_signals/"

gold_df.write \
    .mode("overwrite") \
    .parquet(gold_path)

print("Gold layer written to ADLS successfully")

# Verify by reading back
verify_df = spark.read.parquet(gold_path)
print(f"Verified row count: {verify_df.count()}")

Gold layer written to ADLS successfully
Verified row count: 2000000


In [0]:
# Write gold as CSV for SQL import
csv_path = "abfss://gold@sttelcomchurn.dfs.core.windows.net/fact_churn_csv/"

gold_df.select(
    "customer_id",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "tenure_months",
    "contract_type",
    "monthly_charges",
    "total_charges",
    "payment_method",
    "billing_spike_flag",
    "internet_service",
    "drop_rate",
    "churn_risk_score",
    "churn_risk_label",
    "churned"
).write \
 .mode("overwrite") \
 .option("header", True) \
 .csv(csv_path)

print("Gold CSV written successfully")

Gold CSV written successfully


In [0]:
# ── Load gold data to Azure SQL Database ──────────────────
jdbc_url = (
    "jdbc:sqlserver://sql-telecom-churn.database.windows.net:1433;"
    "database=db-telecom-churn;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user"    : "sqladmin",
    "password": "YOUR_SQL_PASSWORD_HERE",
    "driver"  : "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

gold_df.write \
    .jdbc(
        url        = jdbc_url,
        table      = "fact_churn_signals",
        mode       = "overwrite",
        properties = connection_properties
    )

print("Data loaded to Azure SQL successfully")

Data loaded to Azure SQL successfully
